In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
import pandas as pd 

In [ ]:
cd Downloads

In [ ]:
query="""SELECT *
FROM chat_feedbacks
WHERE ts::date = '2025-11-27';"""

In [ ]:
import psycopg
DB_USER = os.getenv("PGUSER", "")
DB_PASS = os.getenv("PGPASSWORD", "")
DB_NAME = os.getenv("PGDATABASE", "")

DB_HOST = os.getenv("PGHOST", "127.0.0.1")
DB_PORT = 10001          # the port shown by `db-tunnel`
DB_SSL  = os.getenv("PGSSLMODE", "disable")
def pg_conn():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        sslmode="disable",
        
        connect_timeout=10,
    )

In [ ]:
with pg_conn() as con, con.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    colnames = [desc[0] for desc in cur.description]

df = pd.DataFrame(rows, columns=colnames)
df['stars']=df['stars'] +1

In [ ]:
df.to_csv('df.xlsx',index=False)

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="RETEX Analysis Report",
    explorative=True,
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
        "phi_k": {"calculate": True}
    }
)

profile.to_file("retex_profiling_report.html")


In [ ]:
from IPython.display import display, HTML

with open("retex_profiling_report.html", "r", encoding="utf-8") as f:
    html = f.read()

display(HTML(html))


In [ ]:
from textblob import TextBlob

df["sentiment_polarity"] = df["reasons"].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df["sentiment_subjectivity"] = df["reasons"].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)


In [ ]:
import re
import pandas as pd

def clean_text(t):
    if pd.isna(t): 
        return ""
    t = t.lower()
    t = re.sub(r"\s+", " ", t)       # collapse spaces
    t = re.sub(r"[^a-zA-ZÀ-ÿ0-9 ]", " ", t)  # keep alphanum
    return t.strip()

for c in ["reasons", "comment", "question", "answer"]:
    df[c + "_clean"] = df[c].apply(clean_text)


In [ ]:
from textblob import TextBlob

def polarity(t):
    return TextBlob(t).sentiment.polarity

df["sentiment_reason"] = df["reasons_clean"].apply(polarity)
df["sentiment_comment"] = df["comment_clean"].apply(polarity)
df["sentiment_answer"] = df["answer_clean"].apply(polarity)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

texts = (
    df["reasons_clean"]
    .fillna("")           # avoid NaN
    .astype(str)
    .tolist()
)

vec = TfidfVectorizer(   # no more stop_words="french"
    max_df=0.9,
    min_df=2,
    ngram_range=(1, 2)   # unigrams + bigrams, optional but often useful
)

X = vec.fit_transform(texts)

k = 4
kmeans = KMeans(n_clusters=k, random_state=42).fit(X)

df["cluster"] = kmeans.labels_


In [ ]:
FRENCH_STOPWORDS = [
    "alors", "au", "aucun", "aussi", "autre", "avant", "avec",
    "car", "ce", "cela", "ces", "ceux", "chaque", "comme", "comment",
    "dans", "des", "du", "elle", "en", "encore", "entre", "est",
    "et", "eu", "fait", "faites", "fois", "hors", "ici", "il",
    "ils", "je", "là", "la", "le", "les", "leur", "lui", "ma",
    "mais", "me", "même", "mes", "moi", "mon", "ne", "nos",
    "notre", "nous", "on", "ou", "où", "par", "parce", "pas",
    "peu", "plus", "pour", "qu", "que", "quel", "quelle", "quelles",
    "quels", "qui", "sa", "se", "ses", "si", "son", "sont", "sous",
    "sur", "ta", "te", "tes", "toi", "ton", "toujours", "tous",
    "tout", "trop", "très", "tu", "vos", "votre", "vous", "vu",
    "ça", "étaient", "état", "étions", "été", "être"
]

vec = TfidfVectorizer(
    stop_words=FRENCH_STOPWORDS,
    max_df=0.9,
    min_df=2,
    ngram_range=(1, 2),
)
X = vec.fit_transform(texts)


In [ ]:
import matplotlib.pyplot as plt

cluster_counts = df["cluster"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
cluster_counts.plot(kind="bar")
plt.xlabel("Cluster")
plt.ylabel("Nombre de commentaires")
plt.title("Nombre de commentaires par cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

cluster_counts


In [ ]:
import numpy as np

terms = np.array(vec.get_feature_names_out())
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

def show_top_terms_per_cluster(n_terms=10):
    for i in range(k):
        top_terms = terms[order_centroids[i, :n_terms]]
        print(f"Cluster {i}: {', '.join(top_terms)}")

show_top_terms_per_cluster(10)


In [ ]:
import pandas as pd
import re
from textblob import TextBlob

# (décommente si tu lis depuis un fichier)
# df = pd.read_csv("df.csv")  # ou pd.read_excel("df.xlsx")

def clean_text(t):
    if pd.isna(t):
        return ""
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[^a-zA-ZÀ-ÿ0-9 ]", " ", t)
    return t.strip()

df["comment_clean"] = df["comment"].astype(str).apply(clean_text)
df["comment_length"] = df["comment_clean"].apply(lambda x: len(x.split()))

def sentiment_fr(text):
    if not text or text.strip() == "":
        return 0
    return TextBlob(text).sentiment.polarity

df["comment_sentiment"] = df["comment_clean"].apply(sentiment_fr)

df[["comment", "comment_clean", "comment_length", "comment_sentiment"]].head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
df["stars"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Étoiles")
plt.ylabel("Nombre de réponses")
plt.title("Distribution des notes (stars)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
df["comment_sentiment"].plot(kind="hist", bins=20)
plt.xlabel("Sentiment (polarity TextBlob)")
plt.ylabel("Nombre de réponses")
plt.title("Distribution du sentiment des commentaires")
plt.tight_layout()
plt.show()


In [ ]:
sentiment_by_stars = (
    df.groupby("stars")["comment_sentiment"]
      .mean()
      .sort_index()
)

plt.figure(figsize=(6, 4))
sentiment_by_stars.plot(kind="bar")
plt.xlabel("Étoiles")
plt.ylabel("Sentiment moyen du commentaire")
plt.title("Sentiment moyen du commentaire par note (stars)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

sentiment_by_stars
